# Evidencia de transferencia semántica y enrutamiento

El estudio de transferencia cambia las composiciones de tareas y conserva su propio linaje. El estudio de enrutamiento presenta la puerta de asignación ajustada por separado de los resultados de las tareas. Este cuaderno lee ambos informes ligados a sus fuentes y no combina la transferencia con la matriz central.

## Recompute the source-bound summaries

The following cell reads committed evidence only. It does not execute a policy or create new results.

In [1]:
from pathlib import Path
import hashlib, json, os
ROOT = Path(os.environ['NEURAXIS_REPO_ROOT'])
data = ROOT / 'data' / 'derived'
core = json.loads((data / 'index.json').read_text(encoding='utf-8'))
transfer_path = data / 'semantic-transfer' / 'index.json'
transfer_bytes = transfer_path.read_bytes()
transfer = json.loads(transfer_bytes)
report = json.loads((data / 'semantic-transfer' / 'report.json').read_text(encoding='utf-8'))
route = json.loads((data / 'semantic-routing-study' / 'analysis.json').read_text(encoding='utf-8'))
assert hashlib.sha256(transfer_bytes).hexdigest() == report['index_sha256']
metrics = {row['method_id']: row for row in report['methods']}
core_metrics = {row['method_id']: row for row in core['aggregate_results']}
print('El núcleo y la transferencia semántica son conjuntos separados; no se deben combinar sus denominadores.')
print('Episodios de transferencia:', report['episodes_checked'], '| familias:', len(transfer['cases']))
print(f"{'Método':<9} {'Éxitos transferencia':>20} {'Episodios':>10} {'Tasa':>8} {'Tiempo medio ms':>16}")
for method in ('M04', 'M09', 'M12'):
    row = metrics[method]
    print(f"{method:<9} {row['successes']:>20} {row['episodes']:>10} {row['successes']/row['episodes']:>8.3f} {row['mean_wall_ms']:>16.2f}")
print('Resultados pareados de M12:', json.dumps(report['m12_paired'], sort_keys=True))
print('Umbral de la puerta:', route['selected_threshold'], '| positivos de ajuste:', route['fit_positives'], '| positivos de calibración:', route['calibration_positives'])
for label, result in route['test'].items():
    print(f"enrutamiento {label}: episodios={result['episodes']} éxitos_verificados={result['verified_successes']} episodios_planificador={result['planner_episodes']} tiempo_medio_ms={result['mean_wall_ms']:.2f}")
print('Tasas núcleo M04 y M12:', f"{core_metrics['M04']['task_success']:.3f}", f"{core_metrics['M12']['task_success']:.3f}")

El núcleo y la transferencia semántica son conjuntos separados; no se deben combinar sus denominadores.
Episodios de transferencia: 1536 | familias: 8
Método    Éxitos transferencia  Episodios     Tasa  Tiempo medio ms
M04                          2        128    0.016           329.22
M09                         38        128    0.297         18882.47
M12                         18        128    0.141          7706.12
Resultados pareados de M12: {"M01": {"m12_wins": 18, "ties": 110}, "M04": {"m12_losses": 2, "m12_wins": 18, "ties": 108}, "M09": {"m12_losses": 22, "m12_wins": 2, "ties": 104}}
Umbral de la puerta: 0.44629505030857797 | positivos de ajuste: 19 | positivos de calibración: 8
enrutamiento always_fast_M04: episodios=32 éxitos_verificados=0 episodios_planificador=0 tiempo_medio_ms=391.45
enrutamiento always_planner_M09: episodios=32 éxitos_verificados=10 episodios_planificador=32 tiempo_medio_ms=20052.69
enrutamiento hash_half: episodios=32 éxitos_verificados=7 episodios_plan

## Interpretation limits

La transferencia retenida y la pequeña muestra de enrutamiento son evidencia acotada. Un conteo pareado favorable no demuestra transferencia amplia; se deben leer juntos el umbral, el costo del planificador, las salidas inválidas y el denominador.